# 5.1 ***层和块***

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

### 5.1.1 *从零实现一个块*

块的基本功能：  
1、将输入数据作为其前向传播函数的参数。  
2、通过前向传播函数来生成输出。  
3、计算其输出关于输入的梯度，可通过其反向传播函数进行访问。通常这是自动发生的。  
4、存储和访问前向传播计算所需的参数。  
5、根据需要初始化模型参数。

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [ ]:
net = MLP()
X = torch.rand(2, 20)
net(X)

### 5.1.2 *顺序块*

Sequential类至少需要实现：  
1、一种将块逐个追加到列表中的函数。  
2、一种前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”。

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

In [ ]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

# 5.2 ***参数管理***

In [ ]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

### 5.2.1 *参数访问*

从嵌套块访问参数

In [ ]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

In [ ]:
print(rgnet)

### 5.2.2 *参数初始化*

自定义初始化

In [ ]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                         for name, param in m.named_parameters()[0]])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]